# ML-Based IDS Full Pipeline
Operates on the complete NF-UQ-NIDS-v2 dataset (~76M rows) with chunked processing throughout

In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
import joblib
import time
import os
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix,f1_score,precision_score,recall_score
import matplotlib.pyplot as plt
import seaborn as sns

## Config

In [ ]:
CSV_PATH="F:/IDS/IDS/data/NF-UQ-NIDS-v2.csv"
PARQUET_PATH="F:/IDS/IDS/data/NF-UQ-NIDS-v2.parquet"
TRAIN_PARQUET_PATH="train.parquet"
TEST_PARQUET_PATH="test.parquet"
TEST_DEDUP_PATH="test_dedup.parquet"
CONVERT_CHUNK_SIZE=500000
PROCESS_CHUNK_SIZE=5000000
TRAIN_CHUNK_SIZE=12000000
CORR_THRESHOLD=0.95
TOP_K_LIST=[5,10,15,20,None]
MIN_CLASS_COUNT=100
TEST_FRAC=0.1
SCALER_SAMPLE_ROWS=1000000
SWEEP_SAMPLE_ROWS=2000000
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)

## CSV to Parquet conversion

In [ ]:
if not os.path.exists(PARQUET_PATH):
    writer=None
    for chunk in pd.read_csv(CSV_PATH,chunksize=CONVERT_CHUNK_SIZE):
        float_cols=chunk.select_dtypes(include="float64").columns
        chunk[float_cols]=chunk[float_cols].astype("float32")
        table=pa.Table.from_pandas(chunk,preserve_index=False)
        if writer is None:
            writer=pq.ParquetWriter(PARQUET_PATH,table.schema)
        writer.write_table(table)
    writer.close()

## Chunked Parquet reader

In [ ]:
def parquet_chunk_reader(path,columns=None,batch_size=PROCESS_CHUNK_SIZE):
    pf=pq.ParquetFile(path)
    for batch in pf.iter_batches(batch_size=batch_size,columns=columns):
        yield batch.to_pandas()

## Full-dataset class and source tally

In [ ]:
class_counts=pd.Series(dtype="int64")
source_counts=pd.Series(dtype="int64")
for chunk in parquet_chunk_reader(PARQUET_PATH,columns=["Attack","Dataset"]):
    chunk["Attack"]=chunk["Attack"].fillna("Benign")
    class_counts=class_counts.add(chunk["Attack"].value_counts(),fill_value=0)
    source_counts=source_counts.add(chunk["Dataset"].value_counts(),fill_value=0)

class_counts=class_counts.astype("int64")
source_counts=source_counts.astype("int64")
print(class_counts)
print(source_counts)

## Label encoding on classes meeting minimum count

In [ ]:
valid_classes=class_counts[class_counts>=MIN_CLASS_COUNT].index.tolist()
encoder=LabelEncoder()
encoder.fit(valid_classes)
joblib.dump(encoder,"label_encoder.pkl")
print(len(valid_classes),valid_classes)

## Per-source test cutoff

In [ ]:
source_cutoff={source:int(count*(1-TEST_FRAC)) for source,count in source_counts.items()}
print(source_cutoff)

## Cleaning function

In [ ]:
DROP_COLS=["IPV4_SRC_ADDR","IPV4_DST_ADDR","L4_SRC_PORT","L4_DST_PORT","Label","Attack","Dataset"]
def clean_features(x):
    x=x.replace([np.inf,-np.inf],np.nan)
    x=x.fillna(x.median(numeric_only=True))
    x[x<0]=0
    return x

## Per-source time-based split, class filtering and cleaning applied per chunk

In [ ]:
source_seen={source:0 for source in source_counts.index}
train_writer=None
test_writer=None
for chunk in parquet_chunk_reader(PARQUET_PATH,batch_size=PROCESS_CHUNK_SIZE):
    chunk["Attack"]=chunk["Attack"].fillna("Benign")
    chunk=chunk[chunk["Attack"].isin(valid_classes)].reset_index(drop=True)
    if len(chunk)==0:
        continue
    grp_cumcount=chunk.groupby("Dataset").cumcount().values
    source_offset=chunk["Dataset"].map(source_seen).values
    position=source_offset+grp_cumcount
    cutoff_arr=chunk["Dataset"].map(source_cutoff).values
    is_train=position<cutoff_arr
    group_sizes=chunk.groupby("Dataset").size()
    for source,cnt in group_sizes.items():
        source_seen[source]=source_seen.get(source,0)+int(cnt)
    chunk["target"]=encoder.transform(chunk["Attack"])
    feature_cols=[c for c in chunk.columns if c not in DROP_COLS+["target"]]
    chunk[feature_cols]=clean_features(chunk[feature_cols])
    train_chunk=chunk[is_train]
    test_chunk=chunk[~is_train]
    if len(train_chunk)>0:
        table=pa.Table.from_pandas(train_chunk,preserve_index=False)
        if train_writer is None:
            train_writer=pq.ParquetWriter(TRAIN_PARQUET_PATH,table.schema)
        train_writer.write_table(table)
    if len(test_chunk)>0:
        table=pa.Table.from_pandas(test_chunk,preserve_index=False)
        if test_writer is None:
            test_writer=pq.ParquetWriter(TEST_PARQUET_PATH,table.schema)
        test_writer.write_table(table)

if train_writer is not None:
    train_writer.close()
if test_writer is not None:
    test_writer.close()

## Hash all training rows for near-duplicate lookup

In [ ]:
DEDUP_COLS=[c for c in pq.ParquetFile(TRAIN_PARQUET_PATH).schema_arrow.names if c not in DROP_COLS+["target"]]
train_hash_parts=[]
for chunk in parquet_chunk_reader(TRAIN_PARQUET_PATH,columns=DEDUP_COLS,batch_size=PROCESS_CHUNK_SIZE):
    train_hash_parts.append(pd.util.hash_pandas_object(chunk,index=False).values)

train_hashes=np.sort(np.concatenate(train_hash_parts))
print(train_hashes.shape)

## Remove test rows that near-duplicate a training row

In [ ]:
test_writer=None
kept=0
removed=0
for chunk in parquet_chunk_reader(TEST_PARQUET_PATH,batch_size=PROCESS_CHUNK_SIZE):
    chunk_hashes=pd.util.hash_pandas_object(chunk[DEDUP_COLS],index=False).values
    idx=np.searchsorted(train_hashes,chunk_hashes)
    idx=np.clip(idx,0,len(train_hashes)-1)
    is_dup=train_hashes[idx]==chunk_hashes
    keep_mask=~is_dup
    removed+=int(is_dup.sum())
    kept+=int(keep_mask.sum())
    filtered=chunk[keep_mask]
    if len(filtered)>0:
        table=pa.Table.from_pandas(filtered,preserve_index=False)
        if test_writer is None:
            test_writer=pq.ParquetWriter(TEST_DEDUP_PATH,table.schema)
        test_writer.write_table(table)

if test_writer is not None:
    test_writer.close()
print(kept,removed)

## Random sampling helper for fitting/selection steps

In [ ]:
def sample_from_parquet(path,columns,n_target,chunk_size=PROCESS_CHUNK_SIZE,seed=RANDOM_STATE):
    total_rows=pq.ParquetFile(path).metadata.num_rows
    frac=min(1.0,n_target/total_rows*1.2)
    rng=np.random.default_rng(seed)
    parts=[]
    for chunk in parquet_chunk_reader(path,columns=columns,batch_size=chunk_size):
        mask=rng.random(len(chunk))<frac
        parts.append(chunk[mask])
    sample=pd.concat(parts,ignore_index=True)
    if len(sample)>n_target:
        sample=sample.sample(n=n_target,random_state=seed).reset_index(drop=True)
    return sample

## Correlation-based feature removal

In [ ]:
corr_sample=sample_from_parquet(TRAIN_PARQUET_PATH,DEDUP_COLS,SCALER_SAMPLE_ROWS)
corr_matrix=corr_sample.corr().abs()
upper=corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
drop_corr=[col for col in upper.columns if any(upper[col]>CORR_THRESHOLD)]
FEATURES_AFTER_CORR=[c for c in DEDUP_COLS if c not in drop_corr]
joblib.dump(drop_corr,"drop_corr.pkl")
print(drop_corr)

## Fit scaler on training sample

In [ ]:
scaler=StandardScaler()
scaler.fit(corr_sample[FEATURES_AFTER_CORR])
joblib.dump(scaler,"scaler.pkl")

## Feature importance sweep

In [ ]:
sweep_sample=sample_from_parquet(TRAIN_PARQUET_PATH,FEATURES_AFTER_CORR+["target"],SWEEP_SAMPLE_ROWS)
x_sweep=pd.DataFrame(scaler.transform(sweep_sample[FEATURES_AFTER_CORR]),columns=FEATURES_AFTER_CORR)
y_sweep=sweep_sample["target"]
base_model=xgb.XGBClassifier(n_estimators=100,tree_method="hist",random_state=RANDOM_STATE,eval_metric="mlogloss")
base_model.fit(x_sweep,y_sweep)
importances=pd.Series(base_model.feature_importances_,index=x_sweep.columns).sort_values(ascending=False)
sweep_class_counts=y_sweep.value_counts()
can_stratify=sweep_class_counts.min()>=2
split_kwargs={"test_size":0.2,"random_state":RANDOM_STATE}
if can_stratify:
    split_kwargs["stratify"]=y_sweep
x_tr,x_val,y_tr,y_val=train_test_split(x_sweep,y_sweep,**split_kwargs)
best_score=0
best_features=list(x_sweep.columns)
for k in TOP_K_LIST:
    features=list(importances.index) if k is None else list(importances.index[:k])
    model=xgb.XGBClassifier(n_estimators=100,tree_method="hist",random_state=RANDOM_STATE,eval_metric="mlogloss")
    model.fit(x_tr[features],y_tr)
    preds=model.predict(x_val[features])
    score=f1_score(y_val,preds,average="weighted")
    if score>best_score:
        best_score=score
        best_features=features

print("Selected feature count:",len(best_features))
print("Selected features:",best_features)
print("Best validation weighted F1:",best_score)
joblib.dump(best_features,"selected_features.pkl")

## Chunked training data generator

In [ ]:
def get_train_chunks(path,columns,chunk_size):
    for chunk in parquet_chunk_reader(path,columns=columns,batch_size=chunk_size):
        x_full=pd.DataFrame(scaler.transform(chunk[FEATURES_AFTER_CORR]),columns=FEATURES_AFTER_CORR)
        x=x_full[best_features]
        y=chunk["target"]
        yield x,y

train_columns=FEATURES_AFTER_CORR+["target"]

## Chunked continued training - XGBoost

In [ ]:
import gc
from tqdm import tqdm

total_train_rows=pq.ParquetFile(TRAIN_PARQUET_PATH).metadata.num_rows
num_chunks=-(-total_train_rows//TRAIN_CHUNK_SIZE)

xgb_model=None
xgb_train_time=0
xgb_params={"objective":"multi:softprob","num_class":len(encoder.classes_),"tree_method":"hist","eval_metric":"mlogloss","max_bin":128}
progress=tqdm(get_train_chunks(TRAIN_PARQUET_PATH,train_columns,TRAIN_CHUNK_SIZE),total=num_chunks,desc="XGBoost")
for x_chunk,y_chunk in progress:
    start=time.time()
    dtrain=xgb.DMatrix(x_chunk,label=y_chunk)
    xgb_model=xgb.train(xgb_params,dtrain,num_boost_round=100,xgb_model=xgb_model)
    xgb_train_time+=time.time()-start
    xgb_model.save_model("xgb_model_checkpoint.json")
    del x_chunk,y_chunk,dtrain
    gc.collect()

xgb_model.save_model("xgb_model.json")
print(xgb_train_time)

## Chunked continued training - LightGBM

In [ ]:
import gc
from tqdm import tqdm

lgb_model=None
lgb_train_time=0
lgb_params={"objective":"multiclass","num_class":len(encoder.classes_),"verbosity":-1,"max_bin":128}
progress=tqdm(get_train_chunks(TRAIN_PARQUET_PATH,train_columns,TRAIN_CHUNK_SIZE),total=num_chunks,desc="LightGBM")
for x_chunk,y_chunk in progress:
    start=time.time()
    train_set=lgb.Dataset(x_chunk,label=y_chunk)
    lgb_model=lgb.train(lgb_params,train_set,num_boost_round=100,init_model=lgb_model,keep_training_booster=True)
    lgb_train_time+=time.time()-start
    lgb_model.save_model("lgb_model_checkpoint.txt")
    del x_chunk,y_chunk,train_set
    gc.collect()

lgb_model.save_model("lgb_model.txt")
print(lgb_train_time)

## Load frozen, deduplicated test set

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
import joblib
from sklearn.metrics import classification_report,confusion_matrix,f1_score,precision_score,recall_score
import matplotlib.pyplot as plt
import seaborn as sns

TEST_DEDUP_PATH="test_dedup.parquet"
DROP_COLS=["IPV4_SRC_ADDR","IPV4_DST_ADDR","L4_SRC_PORT","L4_DST_PORT","Label","Attack","Dataset"]

encoder=joblib.load("label_encoder.pkl")
scaler=joblib.load("scaler.pkl")
best_features=joblib.load("selected_features.pkl")
drop_corr=joblib.load("drop_corr.pkl")

DEDUP_COLS=[c for c in pq.ParquetFile(TEST_DEDUP_PATH).schema_arrow.names if c not in DROP_COLS+["target"]]
FEATURES_AFTER_CORR=[c for c in DEDUP_COLS if c not in drop_corr]

xgb_model=xgb.Booster()
xgb_model.load_model("xgb_model.json")
lgb_model=lgb.Booster(model_file="lgb_model.txt")

test_columns=FEATURES_AFTER_CORR+["target"]
test_data=pd.read_parquet(TEST_DEDUP_PATH,columns=test_columns)
x_test_full=pd.DataFrame(scaler.transform(test_data[FEATURES_AFTER_CORR]),columns=FEATURES_AFTER_CORR)
x_test_final=x_test_full[best_features]
y_test=test_data["target"]

dtest=xgb.DMatrix(x_test_final)
xgb_probs=xgb_model.predict(dtest)
xgb_preds=np.argmax(xgb_probs,axis=1)

lgb_probs=lgb_model.predict(x_test_final)
lgb_preds=np.argmax(lgb_probs,axis=1)

target_names=encoder.classes_
all_labels=list(range(len(target_names)))
print(classification_report(y_test,xgb_preds,labels=all_labels,target_names=target_names,zero_division=0))
print(classification_report(y_test,lgb_preds,labels=all_labels,target_names=target_names,zero_division=0))

summary=pd.DataFrame({"metric":["weighted_f1","macro_f1","weighted_precision","weighted_recall"],"xgb":[f1_score(y_test,xgb_preds,average="weighted"),f1_score(y_test,xgb_preds,average="macro"),precision_score(y_test,xgb_preds,average="weighted"),recall_score(y_test,xgb_preds,average="weighted")],"lgb":[f1_score(y_test,lgb_preds,average="weighted"),f1_score(y_test,lgb_preds,average="macro"),precision_score(y_test,lgb_preds,average="weighted"),recall_score(y_test,lgb_preds,average="weighted")]})
print(summary)

In [ ]:
test_columns=FEATURES_AFTER_CORR+["target"]
test_data=pd.read_parquet(TEST_DEDUP_PATH,columns=test_columns)
x_test_full=pd.DataFrame(scaler.transform(test_data[FEATURES_AFTER_CORR]),columns=FEATURES_AFTER_CORR)
x_test_final=x_test_full[best_features]
y_test=test_data["target"]
print(x_test_final.shape)

## Inference on test set

In [ ]:
dtest=xgb.DMatrix(x_test_final)
start=time.time()
xgb_probs=xgb_model.predict(dtest)
xgb_infer_time=time.time()-start
xgb_preds=np.argmax(xgb_probs,axis=1)

start=time.time()
lgb_probs=lgb_model.predict(x_test_final)
lgb_infer_time=time.time()-start
lgb_preds=np.argmax(lgb_probs,axis=1)

## Per-class classification report

In [ ]:
target_names=encoder.classes_
all_labels=list(range(len(target_names)))
xgb_report=classification_report(y_test,xgb_preds,labels=all_labels,target_names=target_names,output_dict=True,zero_division=0)
lgb_report=classification_report(y_test,lgb_preds,labels=all_labels,target_names=target_names,output_dict=True,zero_division=0)
print(classification_report(y_test,xgb_preds,labels=all_labels,target_names=target_names,zero_division=0))
print(classification_report(y_test,lgb_preds,labels=all_labels,target_names=target_names,zero_division=0))

## Per-class F1 comparison

In [ ]:
f1_compare=pd.DataFrame({"class":target_names,"xgb_f1":[xgb_report[c]["f1-score"] for c in target_names],"lgb_f1":[lgb_report[c]["f1-score"] for c in target_names]})
f1_compare["winner"]=np.where(f1_compare["xgb_f1"]>f1_compare["lgb_f1"],"xgb","lgb")
print(f1_compare)
f1_compare.plot(x="class",y=["xgb_f1","lgb_f1"],kind="bar",figsize=(10,5))
plt.tight_layout()
plt.show()

## Confusion matrices

In [ ]:
xgb_cm=confusion_matrix(y_test,xgb_preds,labels=all_labels,normalize="true")
lgb_cm=confusion_matrix(y_test,lgb_preds,labels=all_labels,normalize="true")
fig,axes=plt.subplots(1,2,figsize=(16,6))
sns.heatmap(xgb_cm,annot=True,fmt=".2f",xticklabels=target_names,yticklabels=target_names,ax=axes[0])
sns.heatmap(lgb_cm,annot=True,fmt=".2f",xticklabels=target_names,yticklabels=target_names,ax=axes[1])
axes[0].set_title("XGBoost")
axes[1].set_title("LightGBM")
plt.tight_layout()
plt.show()

## False positive rate per class

In [ ]:
def fpr_per_class(y_true,y_pred,classes):
    fprs=[]
    for i in range(len(classes)):
        fp=np.sum((y_pred==i)&(y_true!=i))
        tn=np.sum((y_pred!=i)&(y_true!=i))
        fprs.append(fp/(fp+tn) if (fp+tn)>0 else 0)
    return fprs

xgb_fpr=fpr_per_class(y_test.values,xgb_preds,target_names)
lgb_fpr=fpr_per_class(y_test.values,lgb_preds,target_names)
fpr_table=pd.DataFrame({"class":target_names,"xgb_fpr":xgb_fpr,"lgb_fpr":lgb_fpr})
print(fpr_table)

## Consolidated comparison

In [ ]:
lgb_train_time=6855.331805229187 # Hard coded value for LightGBM training time due to memory constraints during training obtained from training on kaggle notebook
xgb_train_time=13599.62030673027 # Hard coded value for XGBoost training time due to memory constraints during training obtained from training on kaggle notebook
summary=pd.DataFrame({"metric":["weighted_f1","macro_f1","weighted_precision","weighted_recall","train_time_sec","infer_time_sec","avg_fpr"],"xgb":[f1_score(y_test,xgb_preds,average="weighted"),f1_score(y_test,xgb_preds,average="macro"),precision_score(y_test,xgb_preds,average="weighted"),recall_score(y_test,xgb_preds,average="weighted"),xgb_train_time,xgb_infer_time,np.mean(xgb_fpr)],"lgb":[f1_score(y_test,lgb_preds,average="weighted"),f1_score(y_test,lgb_preds,average="macro"),precision_score(y_test,lgb_preds,average="weighted"),recall_score(y_test,lgb_preds,average="weighted"),lgb_train_time,lgb_infer_time,np.mean(lgb_fpr)]})
print(summary)
summary.to_csv("comparison_results.csv",index=False)

## Inference function for new data

In [ ]:
def predict_new_data(raw_df,model_name="xgb"):
    x=raw_df.copy()
    feature_cols=[c for c in x.columns if c not in DROP_COLS]
    x=clean_features(x[feature_cols])
    x=x.drop(columns=[c for c in drop_corr if c in x.columns])
    x_scaled=pd.DataFrame(scaler.transform(x[best_features]),columns=best_features)
    if model_name=="xgb":
        dmat=xgb.DMatrix(x_scaled)
        probs=xgb_model.predict(dmat)
    else:
        probs=lgb_model.predict(x_scaled)
    preds=np.argmax(probs,axis=1)
    labels=encoder.inverse_transform(preds)
    confidence=np.max(probs,axis=1)
    return labels,confidence